# 节点 02：感知机的边界——一道题让 AI 沉睡十年

节点 01 里，感知机用 20 行 Python 自己学会了分类。  
这个 notebook 要展示：**有一道只有 4 行数据的题，感知机永远学不会。**  

然后我们用初中代数，**证明**它为什么学不会。

## Cell 1：准备工具

In [1]:
import sys
import numpy as np
import matplotlib
matplotlib.use('Agg')  # 非交互模式，保证 nbconvert 可执行
import matplotlib.pyplot as plt

# 从 src/ 导入感知机（与 tests/ 测试的是同一份代码，不维护两份实现）
# nbconvert 执行时 CWD = 本 notebook 所在目录，../../src 指向项目根目录的 src/
sys.path.insert(0, '../../src')
from perceptron import Perceptron

print('工具准备好了。')

工具准备好了。


## Cell 2：XOR 数据——只有 4 行

XOR 规则：**两个输入不一样，结果是 1；一样，结果是 0。**

| 输入 A | 输入 B | 答案 |
|--------|--------|------|
|   0    |   0    |  0   |
|   0    |   1    |  1   |
|   1    |   0    |  1   |
|   1    |   1    |  0   |

In [2]:
# XOR 的全部 4 个样本
X_xor = np.array([[0, 0],
                   [0, 1],
                   [1, 0],
                   [1, 1]])
y_xor = np.array([0, 1, 1, 0])  # XOR 答案

print('XOR 数据：')
for (a, b), label in zip(X_xor, y_xor):
    print(f'  A={a}, B={b}  →  答案={label}')

XOR 数据：
  A=0, B=0  →  答案=0
  A=0, B=1  →  答案=1
  A=1, B=0  →  答案=1
  A=1, B=1  →  答案=0


## Cell 3：让感知机学 XOR

**在看结果之前，先猜一猜**：感知机训练 500 轮，最终准确率会是多少？

- 猜 100%？
- 猜 75%（4 个答对 3 个）？
- 猜永远卡住？

In [3]:
p = Perceptron(n_features=2, learning_rate=0.1)
p.fit(X_xor, y_xor, max_epochs=500)

# 看看每一轮的错误数
print('部分轮次的错误数：')
for epoch in [0, 49, 99, 199, 499]:
    print(f'  第 {epoch+1:3d} 轮 | 错误: {p.history[epoch]}')

# 计算最终准确率
correct = sum(p.predict(xi) == yi for xi, yi in zip(X_xor, y_xor))
accuracy = correct / len(y_xor)
print(f'\n最终准确率：{accuracy:.0%}（{correct}/{len(y_xor)} 个答对）')

if accuracy < 1.0:
    print('→ 感知机没有收敛。永远卡住了。')

部分轮次的错误数：
  第   1 轮 | 错误: 2
  第  50 轮 | 错误: 4
  第 100 轮 | 错误: 4
  第 200 轮 | 错误: 4
  第 500 轮 | 错误: 4

最终准确率：50%（2/4 个答对）
→ 感知机没有收敛。永远卡住了。


## Cell 4：为什么学不会？——用眼睛看

把 4 个 XOR 点画出来，问一个简单的问题：**能画一条直线，把 ○（答案=1）和 ●（答案=0）分开吗？**

In [4]:
fig, ax = plt.subplots(figsize=(5, 5))

# 画 4 个点（答案=0 用实心黑点，答案=1 用空心圆）
for (a, b), label in zip(X_xor, y_xor):
    if label == 1:
        ax.scatter(a, b, s=200, color='white', edgecolor='black',
                   linewidth=2, zorder=3, label='答案=1 (○)')
    else:
        ax.scatter(a, b, s=200, color='black', zorder=3, label='答案=0 (●)')
    ax.annotate(f'({a},{b})→{label}', (a, b),
                textcoords='offset points', xytext=(10, 5), fontsize=11)

ax.set_xlim(-0.5, 1.5)
ax.set_ylim(-0.5, 1.5)
ax.set_xlabel('A', fontsize=13)
ax.set_ylabel('B', fontsize=13)
ax.set_title('XOR 的 4 个点\n能画一条直线把 ○ 和 ● 分开吗？', fontsize=13)
ax.grid(True, alpha=0.3)

# 去除重复图例
handles, labels = ax.get_legend_handles_labels()
seen = {}
for h, l in zip(handles, labels):
    if l not in seen:
        seen[l] = h
ax.legend(seen.values(), seen.keys(), fontsize=11)

plt.tight_layout()
plt.savefig('xor_plot.png', dpi=100)
plt.close()
print('图已保存为 xor_plot.png')
print()
print('观察：答案=1 的两个点在对角线上 (0,1) 和 (1,0)')
print('      答案=0 的两个点在另一条对角线上 (0,0) 和 (1,1)')
print('      不管怎么画直线，都会有点站错边。')
print('      这叫「线性不可分」。')

图已保存为 xor_plot.png

观察：答案=1 的两个点在对角线上 (0,1) 和 (1,0)
      答案=0 的两个点在另一条对角线上 (0,0) 和 (1,1)
      不管怎么画直线，都会有点站错边。
      这叫「线性不可分」。


/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_24154/1759973826.py:28: UserWarning: Glyph 30340 (\N{CJK UNIFIED IDEOGRAPH-7684}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_24154/1759973826.py:28: UserWarning: Glyph 20010 (\N{CJK UNIFIED IDEOGRAPH-4E2A}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_24154/1759973826.py:28: UserWarning: Glyph 28857 (\N{CJK UNIFIED IDEOGRAPH-70B9}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_24154/1759973826.py:28: UserWarning: Glyph 33021 (\N{CJK UNIFIED IDEOGRAPH-80FD}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_24154/1759973826.py:28: UserWarning: Glyph 30011 (\N{CJK UNIFIED IDEOGRAPH-753B}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/f7/ly4g

## Cell 5：用代数证明——为什么绝对不可能

感知机的判断规则是一条直线方程：  
`w₁ × A + w₂ × B + bias > 0  →  输出 1`

如果感知机能学会 XOR，那必须存在某组 (w₁, w₂, bias) 让下面 4 条同时成立：

| 点 | A | B | 答案 | 要求 |
|----|---|---|------|------|
| ① | 0 | 0 |  0   | `w₁×0 + w₂×0 + bias ≤ 0`，即 `bias ≤ 0` |
| ② | 0 | 1 |  1   | `w₁×0 + w₂×1 + bias > 0`，即 `w₂ + bias > 0` |
| ③ | 1 | 0 |  1   | `w₁×1 + w₂×0 + bias > 0`，即 `w₁ + bias > 0` |
| ④ | 1 | 1 |  0   | `w₁×1 + w₂×1 + bias ≤ 0`，即 `w₁ + w₂ + bias ≤ 0` |

**用代数推下去**：把 ② 和 ③ 相加：  
`(w₂ + bias) + (w₁ + bias) > 0`  
`w₁ + w₂ + 2×bias > 0`  

但由 ① 知 `bias ≤ 0`，所以：  
`w₁ + w₂ + 2×bias ≤ w₁ + w₂ + bias`  

而 ④ 告诉我们 `w₁ + w₂ + bias ≤ 0`，因此：  
`w₁ + w₂ + 2×bias ≤ 0`  

这和我们推出的 `w₁ + w₂ + 2×bias > 0` 矛盾！

**结论**：不存在任何 (w₁, w₂, bias) 能同时满足这 4 条要求。感知机学不会 XOR，不是因为运气差——是数学上不可能。

In [5]:
# 用代码验证：穷举大量 (w1, w2, bias) 组合，找不到一个能学会 XOR
def check_linear_separability(X, y, n_trials=100_000, rng_seed=42):
    """随机搜索线性分类器，检验是否线性可分。"""
    rng = np.random.default_rng(rng_seed)
    for _ in range(n_trials):
        w = rng.uniform(-5, 5, size=2)
        b = rng.uniform(-5, 5)
        preds = (X @ w + b > 0).astype(int)
        if np.all(preds == y):
            return True, w, b
    return False, None, None

separable, w_found, b_found = check_linear_separability(X_xor, y_xor)

if separable:
    print(f'找到了线性分类器：w={w_found}, bias={b_found}')
else:
    print(f'穷举 100,000 种 (w₁, w₂, bias) 组合，没有一个能正确分类所有 XOR 样本。')
    print('XOR 是线性不可分的。')

穷举 100,000 种 (w₁, w₂, bias) 组合，没有一个能正确分类所有 XOR 样本。
XOR 是线性不可分的。


## Cell 6：历史影响——一本书如何让 AI 沉睡十年

1969 年，Minsky & Papert 把这个证明写成了书：  
*Perceptrons: An Introduction to Computational Geometry*（MIT Press）

书里的结论传递出一个信号：**感知机不只是学不会 XOR，它学不了很多「有趣」的问题。**

结果是：  
- 研究经费开始缩减  
- 神经网络论文发表数量暴跌  
- 大约 1969–1982 年：AI 进入「寒冬」

**但这个证明其实有一个很重要的限定语**：

> 单层感知机学不会 XOR。

「单层」是关键。如果叠两层呢？答案是下一个节点的故事（1986 年，反向传播重新激活了这个领域）。

In [6]:
# 画一张感知机学习曲线图：展示「永远卡住」的直观证据
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, len(p.history) + 1), p.history, color='firebrick', linewidth=1.5)
ax.axhline(y=0, color='green', linestyle='--', linewidth=1.5, label='目标：0 个错误')
ax.set_xlabel('训练轮次', fontsize=12)
ax.set_ylabel('本轮错误数', fontsize=12)
ax.set_title('感知机训练 XOR — 500 轮都无法收敛', fontsize=13)
ax.legend(fontsize=11)
ax.set_ylim(-0.1, 4.5)
plt.tight_layout()
plt.savefig('xor_training.png', dpi=100)
plt.close()
print('学习曲线已保存为 xor_training.png')

min_errors = min(p.history)
print(f'\n500 轮中最少错误数：{min_errors}')
print(f'感知机始终无法学会 XOR（错误数从未降到 0）。')

学习曲线已保存为 xor_training.png

500 轮中最少错误数：2
感知机始终无法学会 XOR（错误数从未降到 0）。


/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_24154/1673715138.py:10: UserWarning: Glyph 35757 (\N{CJK UNIFIED IDEOGRAPH-8BAD}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_24154/1673715138.py:10: UserWarning: Glyph 32451 (\N{CJK UNIFIED IDEOGRAPH-7EC3}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_24154/1673715138.py:10: UserWarning: Glyph 36718 (\N{CJK UNIFIED IDEOGRAPH-8F6E}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_24154/1673715138.py:10: UserWarning: Glyph 27425 (\N{CJK UNIFIED IDEOGRAPH-6B21}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_24154/1673715138.py:10: UserWarning: Glyph 26412 (\N{CJK UNIFIED IDEOGRAPH-672C}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/f7/ly4g

## 总结

| 问题 | 感知机能解决吗？ | 为什么 |
|------|-----------------|--------|
| 线性可分问题（如节点 01 的分类）| ✅ 是 | 存在直线可以分开 |
| XOR | ❌ 否 | 数学上证明不可能 |

**关键洞见**：感知机是一条直线。如果问题能被直线分开，它就能学会；不能，它就永远学不会。

**历史结果**：这个发现让 AI 研究冷却了约 13 年。直到 1986 年有人想出了「加一层」的方法。

---
*参考：Minsky & Papert (1969), Perceptrons, MIT Press, ISBN 9780262630221*